In [1]:
import warnings
warnings.filterwarnings("ignore")      # oculta avisos técnicos para que la pantalla quede limpia

import numpy as np                     # cálculo numérico
import pandas as pd                    # tablas de datos
import matplotlib.pyplot as plt        # gráficas
import kagglehub                       # descarga datasets de Kaggle

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay, confusion_matrix,
                             accuracy_score, recall_score, precision_score, f1_score)

---
## 1 · Los datos

**Heart Failure Prediction Dataset** (fedesoriano, Kaggle): 918 pacientes de cinco cohortes cardiológicas.

| Variable | Significado clínico |
|---|---|
| `Age` | Edad (años) |
| `Sex` | Sexo (M / F) |
| `ChestPainType` | Dolor torácico: TA angina típica · ATA atípica · NAP no anginoso · ASY asintomático |
| `RestingBP` | Presión arterial sistólica en reposo (mm Hg) |
| `Cholesterol` | Colesterol sérico (mg/dL) |
| `FastingBS` | Glucosa en ayuno > 120 mg/dL (1 = sí) |
| `RestingECG` | ECG en reposo: Normal · ST (anomalía ST-T) · LVH (hipertrofia ventricular izquierda) |
| `MaxHR` | Frecuencia cardiaca máxima en prueba de esfuerzo |
| `ExerciseAngina` | Angina inducida por el ejercicio (Y / N) |
| `Oldpeak` | Depresión del segmento ST con el esfuerzo (mm) |
| `ST_Slope` | Pendiente del ST en el pico del esfuerzo: Up · Flat · Down |
| **`HeartDisease`** | **Lo que queremos predecir: 1 = con enfermedad cardiaca · 0 = sin enfermedad** |

In [2]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
df = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS,
                            "fedesoriano/heart-failure-prediction", "heart.csv")
#Importando datos

In [3]:
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [4]:
#Imprimiendo renglones y columnas del dataframe
df.shape

(918, 12)

# Preparacion de los datos

In [5]:
#Imprimiendo tipos de datos
df.dtypes

Age                 int64
Sex                   str
ChestPainType         str
RestingBP           int64
Cholesterol         int64
FastingBS           int64
RestingECG            str
MaxHR               int64
ExerciseAngina        str
Oldpeak           float64
ST_Slope              str
HeartDisease        int64
dtype: object

In [6]:
#Comprobando si existen datos nulos
df.isna().sum()

Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0
dtype: int64

In [7]:
#Explorando los datos numericos
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,918.0,53.510893,9.432617,28.0,47.00,54.0,60.0,77.0
RestingBP,918.0,132.396514,18.514154,0.0,120.00,130.0,140.0,200.0
Cholesterol,918.0,198.799564,109.384145,0.0,173.25,223.0,267.0,603.0
FastingBS,918.0,0.233115,0.423046,0.0,0.00,0.0,0.0,1.0
MaxHR,918.0,136.809368,25.460334,60.0,120.00,138.0,156.0,202.0
Oldpeak,918.0,0.887364,1.066570,-2.6,0.00,0.6,1.5,6.2
HeartDisease,918.0,0.553377,0.497414,0.0,0.00,1.0,1.0,1.0


In [8]:
#Corrigiendo los valores numericos en 0 que no hacen sentido  
df['Cholesterol'] = df['Cholesterol'].replace(0, np.nan)
df['RestingBP'] = df['RestingBP'].replace(0, np.nan)

In [9]:
#Validando la conversion a valores nulos
df.isna().sum()

Age                 0
Sex                 0
ChestPainType       0
RestingBP           1
Cholesterol       172
FastingBS           0
RestingECG          0
MaxHR               0
ExerciseAngina      0
Oldpeak             0
ST_Slope            0
HeartDisease        0
dtype: int64

# Dividiendo los datos

In [10]:
# Separando los datos en x e y
y = df['HeartDisease']
X= df.drop(columns=['HeartDisease'])

In [11]:
#Definiendo columnas de tipo string
string_columns = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']

#Iterando sobre las columnas string 
for column in string_columns:
    print(X[column].value_counts())
    print("\n")

Sex
M    725
F    193
Name: count, dtype: int64


ChestPainType
ASY    496
NAP    203
ATA    173
TA      46
Name: count, dtype: int64


RestingECG
Normal    552
LVH       188
ST        178
Name: count, dtype: int64


ExerciseAngina
N    547
Y    371
Name: count, dtype: int64


ST_Slope
Flat    460
Up      395
Down     63
Name: count, dtype: int64




In [ ]:
print(X.head())
print(X.shape)

In [ ]:
#Aplicando One_Hot a las columnas string
X = pd.get_dummies(X, drop_first=True, dtype=int)
X.head()